# CVE Notebook

In [11]:
import polars as pl
from IPython.display import display


df = pl.read_parquet("../www/static/gen/data/test.parquet")
print(pl.read_parquet_schema("../www/static/gen/data/test.parquet"))
# display(df.to_pandas())
df_with_year = df.with_columns(
    pl.col("id").str.extract(r"CVE-(\d{4})").cast(pl.Int64).alias("year")
)
display(df_with_year)

# count cve per assigner per year
cve_per_assigner_year = (
    df_with_year.group_by(["year", "assigner"])
    .agg(pl.count("id").alias("cve_count"))
    .sort(["year", "cve_count"], descending=[False, True])
)
display(cve_per_assigner_year)

# only show for 2025

# get list of assigners present in the df
assigners = df_with_year.select(pl.col("assigner").unique().sort()).to_series()
print(assigners)


{'id': String, 'assigner': String, 'state': String, 'cvss_v2': Float64, 'cvss_v3': Float64, 'cvss_v3_1': Float64, 'cvss_v4': Float64, 'adp_cvss_v2': Float64, 'adp_cvss_v3': Float64, 'adp_cvss_v3_1': Float64, 'adp_cvss_v4': Float64, 'date_reserved': String, 'date_published': String, 'cna_provider': String}


id,assigner,state,cvss_v2,cvss_v3,cvss_v3_1,cvss_v4,adp_cvss_v2,adp_cvss_v3,adp_cvss_v3_1,adp_cvss_v4,date_reserved,date_published,cna_provider,year
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str,i64
"""CVE-1999-0001""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""2000-02-04T05:00:00""","""mitre""",1999
"""CVE-1999-0002""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""1999-09-29T04:00:00""","""mitre""",1999
"""CVE-1999-0003""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""1999-09-29T04:00:00""","""mitre""",1999
"""CVE-1999-0004""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""2000-02-04T05:00:00""","""mitre""",1999
"""CVE-1999-0005""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""1999-09-29T04:00:00""","""mitre""",1999
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CVE-2025-9994""","""certcc""","""PUBLISHED""",null,null,null,null,null,null,9.8,null,"""2025-09-04T15:31:44.614Z""","""2025-09-09T13:01:05.384Z""","""certcc""",2025
"""CVE-2025-9996""","""schneider""","""PUBLISHED""",null,null,null,5.8,null,null,null,null,"""2025-09-04T16:16:03.592Z""","""2025-09-09T21:11:15.034Z""","""schneider""",2025
"""CVE-2025-9997""","""schneider""","""PUBLISHED""",null,null,null,5.8,null,null,null,null,"""2025-09-04T16:16:04.091Z""","""2025-09-09T21:12:35.969Z""","""schneider""",2025


year,assigner,cve_count
i64,str,u32
1999,"""mitre""",1574
1999,"""redhat""",5
2000,"""mitre""",1237
2000,"""redhat""",5
2001,"""mitre""",1540
…,…,…
2025,"""YokogawaGroup""",1
2025,"""OB""",1
2025,"""Forescout""",1


shape: (443,)
Series: 'assigner' [str]
[
	"1E"
	"360ST"
	"3DS"
	"42Gears"
	"9front"
	…
	"wolfSSL"
	"yandex"
	"zdi"
	"zephyr"
	"zte"
]


In [12]:
# remove entries that have date_published as null
df_with_year = df_with_year.filter(pl.col("date_published").is_not_null())

# truncate the "date_published" column so we only keep the year and the month ("2025-07-12T09:24:28.763Z" -> 2025-07)
df_with_year = df_with_year.with_columns(
    pl.col("date_published").str.slice(0, 7).alias("date")
)
print(df_with_year)

# sum the CVEs per month per assigner
cve_per_month_publisher = (
    df_with_year.group_by(["date", "assigner"])
    .agg(pl.count("id").alias("cve_count"))
    .sort(["date", "cve_count"], descending=[False, True])
)
display(cve_per_month_publisher)

# rename cve_count to value, rename assigner to name
cve_per_month_publisher = cve_per_month_publisher.rename({"cve_count": "value", "assigner": "name"})

# sort by assigner and date to ensure correct cumulative sum
cve_per_month_publisher = cve_per_month_publisher.sort(["name", "date"])

# compute cumulative sum per assigner (name) using pl.col().cumsum().over()
cve_per_month_publisher = cve_per_month_publisher.with_columns(
    pl.col("value").cum_sum().over("name").alias("cumulative_value")
)
# only keep the cumulative_value and date columns
cve_per_month_publisher = cve_per_month_publisher.select(["date", "name", "cumulative_value"])

# display the result
display(cve_per_month_publisher)

# export the cumulative sum to csv
cve_per_month_publisher = cve_per_month_publisher.sort("date")

# Export cumulative but per year
cve_per_year_publisher = (
    df_with_year.group_by(["year", "assigner"])
    .agg(pl.count("id").alias("cve_count"))
    .sort(["year", "cve_count"], descending=[False, True])
)
cve_per_year_publisher = cve_per_year_publisher.rename({"cve_count": "value", "assigner": "name"})

cve_per_year_publisher = cve_per_year_publisher.with_columns(
    (pl.col("year").cast(pl.Utf8) + "-01").alias("year")
)
cve_per_year_publisher = cve_per_year_publisher.with_columns(
    pl.col("value").cum_sum().over("name").alias("value")
)
cve_per_year_publisher = cve_per_year_publisher.select(["year", "name", "value"])
display(cve_per_year_publisher)




shape: (320_139, 16)
┌──────────────┬───────────┬───────────┬─────────┬───┬──────────────┬─────────────┬──────┬─────────┐
│ id           ┆ assigner  ┆ state     ┆ cvss_v2 ┆ … ┆ date_publish ┆ cna_provide ┆ year ┆ date    │
│ ---          ┆ ---       ┆ ---       ┆ ---     ┆   ┆ ed           ┆ r           ┆ ---  ┆ ---     │
│ str          ┆ str       ┆ str       ┆ f64     ┆   ┆ ---          ┆ ---         ┆ i64  ┆ str     │
│              ┆           ┆           ┆         ┆   ┆ str          ┆ str         ┆      ┆         │
╞══════════════╪═══════════╪═══════════╪═════════╪═══╪══════════════╪═════════════╪══════╪═════════╡
│ CVE-1999-000 ┆ mitre     ┆ PUBLISHED ┆ null    ┆ … ┆ 2000-02-04T0 ┆ mitre       ┆ 1999 ┆ 2000-02 │
│ 1            ┆           ┆           ┆         ┆   ┆ 5:00:00      ┆             ┆      ┆         │
│ CVE-1999-000 ┆ mitre     ┆ PUBLISHED ┆ null    ┆ … ┆ 1999-09-29T0 ┆ mitre       ┆ 1999 ┆ 1999-09 │
│ 2            ┆           ┆           ┆         ┆   ┆ 4:00:00      ┆ 

date,assigner,cve_count
str,str,u32
"""1999-09""","""mitre""",321
"""2000-01""","""mitre""",182
"""2000-02""","""mitre""",407
"""2000-03""","""mitre""",70
"""2000-04""","""mitre""",112
…,…,…
"""2025-12""","""EDB""",1
"""2025-12""","""Wiz""",1
"""2025-12""","""Moxa""",1


date,name,cumulative_value
str,str,u32
"""2023-10""","""1E""",3
"""2023-11""","""1E""",6
"""2024-08""","""1E""",7
"""2025-03""","""1E""",8
"""2019-11""","""360ST""",1
…,…,…
"""2025-04""","""zte""",185
"""2025-08""","""zte""",186
"""2025-09""","""zte""",188


year,name,value
str,str,u32
"""1999-01""","""mitre""",1574
"""1999-01""","""redhat""",5
"""2000-01""","""mitre""",2811
"""2000-01""","""redhat""",10
"""2001-01""","""mitre""",4351
…,…,…
"""2025-01""","""NX""",2
"""2025-01""","""iManage""",1
"""2025-01""","""PaperCut""",14


## CVE Per Publisher per year no MITRE (>2015)

In [13]:
# Cumulative per year, but excluding mitre and from 2015 only
cve_per_year_publisher_no_mitre = (
    df_with_year
    .filter((pl.col("assigner") != "mitre") & (pl.col("year") >= 2015))
    .group_by(["year", "assigner"])
    .agg(pl.count("id").alias("value"))
    .sort(["assigner", "year"])
    .rename({"assigner": "name"})
    .with_columns(
        (pl.col("year").cast(pl.Utf8) + "-01").alias("year")
    )
    .with_columns(
        pl.col("value").cum_sum().over("name").alias("value")
    )
    .select(["year", "name", "value"])
    .sort("year")
)


display(cve_per_year_publisher_no_mitre)
cve_per_year_publisher_no_mitre.write_csv("../www/static/gen/data/data_cumulative_year_no_mitre_2015.csv")

year,name,value
str,str,u32
"""2015-01""","""CERTVDE""",1
"""2015-01""","""Chrome""",151
"""2015-01""","""Go""",1
"""2015-01""","""VulDB""",128
"""2015-01""","""VulnCheck""",2
…,…,…
"""2025-01""","""wolfSSL""",28
"""2025-01""","""yandex""",38
"""2025-01""","""zdi""",3197


## Cumulative sum of CVEs

In [14]:
# Compute cumulative sum of CVEs per month (across all assigners)
total_cves_per_month = (
    df_with_year
    .group_by("date")
    .agg(pl.count("id").alias("value"))
    .sort("date")
    .with_columns(
        # Ensure date is in YYYY-MM-DD format (add "-01" for day)
        (pl.col("date") + "-01").alias("date"),
        pl.col("value").cum_sum().alias("value")
    )
    .select(["date", "value"])
)

print(total_cves_per_month)

# Export to CSV
total_cves_per_month.write_csv("../www/static/gen/data/total_cves_per_month_cumulative.csv")

print(df)

# Compute cumulative sum of CVEs per month where state == "PUBLISHED"
total_cves_per_month_published = (
    df_with_year
    .filter(pl.col("state") == "PUBLISHED")
    .group_by("date")
    .agg(pl.count("id").alias("value"))
    .sort("date")
    .with_columns(
        (pl.col("date") + "-01").alias("date"),
        pl.col("value").cum_sum().alias("value")
    )
    .select(["date", "value"])
)

print(total_cves_per_month_published)

# Export to CSV
total_cves_per_month_published.write_csv("../www/static/gen/data/total_cves_per_month_published_cumulative.csv")





shape: (311, 2)
┌────────────┬────────┐
│ date       ┆ value  │
│ ---        ┆ ---    │
│ str        ┆ u32    │
╞════════════╪════════╡
│ 1999-09-01 ┆ 321    │
│ 2000-01-01 ┆ 503    │
│ 2000-02-01 ┆ 910    │
│ 2000-03-01 ┆ 980    │
│ 2000-04-01 ┆ 1092   │
│ …          ┆ …      │
│ 2025-08-01 ┆ 303563 │
│ 2025-09-01 ┆ 307905 │
│ 2025-10-01 ┆ 312206 │
│ 2025-11-01 ┆ 315242 │
│ 2025-12-01 ┆ 320139 │
└────────────┴────────┘
shape: (324_630, 14)
┌────────────┬───────────┬───────────┬─────────┬───┬───────────┬───────────┬───────────┬───────────┐
│ id         ┆ assigner  ┆ state     ┆ cvss_v2 ┆ … ┆ adp_cvss_ ┆ date_rese ┆ date_publ ┆ cna_provi │
│ ---        ┆ ---       ┆ ---       ┆ ---     ┆   ┆ v4        ┆ rved      ┆ ished     ┆ der       │
│ str        ┆ str       ┆ str       ┆ f64     ┆   ┆ ---       ┆ ---       ┆ ---       ┆ ---       │
│            ┆           ┆           ┆         ┆   ┆ f64       ┆ str       ┆ str       ┆ str       │
╞════════════╪═══════════╪═══════════╪═════════╪══

## CWE stuff

In [15]:
# Load the CVE-CWE mapping parquet file
cve_cwe_df = pl.read_parquet("../www/static/gen/data/cve_cwe.parquet")

# Compute the number of CVEs per CWE
cve_count_per_cwe = (
    cve_cwe_df.group_by("cwe")
    .agg(pl.count("cve_id").alias("cve_count"))
    .sort("cve_count", descending=True)
    .select(["cwe", "cve_count"])
)
display(cve_count_per_cwe)

cwe,cve_count
str,u32
"""CWE-79""",21088
"""CWE-89""",9354
"""CWE-352""",4719
"""CWE-862""",4407
"""CWE-787""",3764
…,…
"""CWE-293""",1
"""CWE-485""",1
"""CWE-309""",1


In [16]:
# Aggregate the count of CVEs per CWE per year and export for datarace.html
import polars as pl
df = pl.read_parquet("../www/static/gen/data/test.parquet")
cve_cwe_df = pl.read_parquet("../www/static/gen/data/cve_cwe.parquet")

# Merge to get year for each CVE-CWE pair
cve_cwe_year_df = cve_cwe_df.join(df.select(["id", "date_published"]).with_columns([pl.col("id").alias("cve_id")]), on="cve_id", how="left")

# Extract year from date_published (assume format YYYY-MM-DD or YYYY-MM)
cve_cwe_year_df = cve_cwe_year_df.with_columns([pl.col("date_published").str.slice(0,4).cast(pl.Int64).alias("year")])

# Group by CWE and year, count CVEs
cve_count_per_cwe_year = (
    cve_cwe_year_df.group_by(["cwe", "year"])
    .agg(pl.count("cve_id").alias("value"))
    .sort(["year", "value"], descending=[False, True])
    .with_columns([
        (pl.col("year").cast(pl.Utf8) + "-01").alias("date"),
        pl.col("cwe").alias("name")
    ])
    .select(["date", "name", "value"])
)

display(cve_count_per_cwe_year)

# Export to CSV for datarace.html
cve_count_per_cwe_year.write_csv("../www/static/gen/data/cve_per_cwe_per_year.csv")

date,name,value
str,str,u32
"""1999-01""","""CWE-125""",3
"""1999-01""","""CWE-200""",2
"""1999-01""","""CWE-1067""",1
"""1999-01""","""CWE-364""",1
"""1999-01""","""CWE-522""",1
…,…,…
"""2025-01""","""CWE-649""",1
"""2025-01""","""CWE-790""",1
"""2025-01""","""CWE-30""",1


## Products

In [17]:
df = pl.read_parquet("../www/static/gen/data/cve_affected_products.parquet")

display(df)

cve_id,vendor,product
str,str,str
"""CVE-1999-0001""","""n/a""","""n/a"""
"""CVE-1999-0002""","""n/a""","""n/a"""
"""CVE-1999-0003""","""n/a""","""n/a"""
"""CVE-1999-0004""","""n/a""","""n/a"""
"""CVE-1999-0005""","""n/a""","""n/a"""
…,…,…
"""CVE-2025-9996""","""Schneider Electric""","""Saitel DP RTU"""
"""CVE-2025-9997""","""Schneider Electric""","""Saitel DR RTU"""
"""CVE-2025-9997""","""Schneider Electric""","""Saitel DP RTU"""


In [25]:
# Load affected products parquet file
affected_products_df = pl.read_parquet("../www/static/gen/data/cve_affected_products.parquet")

# Load the main CVE dataframe to get provider information
cve_df = pl.read_parquet("../www/static/gen/data/test.parquet")

# Extract year from CVE ID and filter for 2020 and later
cve_df = cve_df.with_columns(
    pl.col("id").str.extract(r"CVE-(\d{4})").cast(pl.Int64).alias("year")
).filter(pl.col("year") >= 2020)

# Join affected products with CVE data to get cna_provider for each CVE
products_with_provider = (
    affected_products_df
    .join(cve_df.select(["id", "cna_provider", "year"]), left_on="cve_id", right_on="id", how="inner")
)

# Count product occurrences per provider
product_count_per_provider = (
    products_with_provider
    .group_by(["cna_provider", "vendor", "product"])
    .agg(pl.col("cve_id").n_unique().alias("cve_count"))
    .sort("cve_count", descending=True)
)

display(product_count_per_provider)

# Also show total CVEs per product across all providers (2020+)
total_product_count = (
    products_with_provider
    .group_by(["vendor", "product"])
    .agg(pl.col("cve_id").n_unique().alias("cve_count"))
    .sort("cve_count", descending=True)
)

print("\nTop products by CVE count (across all providers, 2020+):")
display(total_product_count)

# Show provider-level summary
provider_summary = (
    products_with_provider
    .group_by("cna_provider")
    .agg([
        pl.col("cve_id").n_unique().alias("unique_cves"),
        pl.col("product").n_unique().alias("unique_products"),
        pl.col("vendor").n_unique().alias("unique_vendors")
    ])
    .sort("unique_cves", descending=True)
)

print("\nSummary by provider (2020+):")
display(provider_summary)


cna_provider,vendor,product,cve_count
str,str,str,u32
"""mitre""","""n/a""","""n/a""",35103
"""Linux""","""Linux""","""Linux""",9737
"""microsoft""","""Microsoft""","""Windows Server 2019""",3084
"""microsoft""","""Microsoft""","""Windows Server 2019 (Server Co…",3061
"""microsoft""","""Microsoft""","""Windows 10 Version 1809""",2804
…,…,…,…
"""Wordfence""","""magazine3""","""Web Stories Enhancer – Level U…",1
"""Wordfence""","""daveshine""","""Toolbar Extras for Elementor &…",1
"""MediaTek""","""MediaTek, Inc.""","""MT6595, MT6735, MT6737, MT6739…",1



Top products by CVE count (across all providers, 2020+):


vendor,product,cve_count
str,str,u32
"""n/a""","""n/a""",35319
"""Linux""","""Linux""",9756
"""Microsoft""","""Windows Server 2019""",3086
"""Microsoft""","""Windows Server 2019 (Server Co…",3061
"""Microsoft""","""Windows 10 Version 1809""",2804
…,…,…
"""dev4press""","""coreactivity""",1
"""aditya_infotech_limited""","""cp-uvr-0801k1b-h""",1
"""rtpHarry""","""Bulk Page Stub Creator""",1



Summary by provider (2020+):


cna_provider,unique_cves,unique_products,unique_vendors
str,u32,u32,u32
"""mitre""",35844,3271,2226
"""Patchstack""",13896,10290,7213
"""VulDB""",10960,4086,1811
"""Linux""",9737,4,2
"""GitHub_M""",9595,3118,2573
…,…,…,…
"""alibaba""",1,1,1
"""Tanium""",1,1,1
"""ELAN""",1,2,2


In [19]:
# Count unique CVEs per product (ignore vendor)
cve_per_product = (
    df
    .group_by("product")
    .agg(pl.col("cve_id").n_unique().alias("cve_count"))
    .sort("cve_count", descending=True)
)

# Count unique CVEs per vendor+product pair (more precise)
cve_per_vendor_product = (
    df
    .group_by(["vendor", "product"])
    .agg(pl.col("cve_id").n_unique().alias("cve_count"))
    .sort("cve_count", descending=True)
)

# Show top results
display(cve_per_product.head(50))
display(cve_per_vendor_product.head(50))

# Optionally save full results
cve_per_product.write_csv("../www/static/gen/data/top_products_by_cves.csv")
cve_per_vendor_product.write_csv("../www/static/gen/data/top_vendor_product_by_cves.csv")

product,cve_count
str,u32
"""n/a""",141544
"""Linux""",9769
"""Android""",4580
"""Windows Server 2019""",3263
"""Windows Server 2019 (Server Co…",3167
…,…
"""Windows 7""",922
"""Windows 11 Version 24H2""",917
"""Thunderbird""",901


vendor,product,cve_count
str,str,u32
"""n/a""","""n/a""",141542
"""Linux""","""Linux""",9760
"""Microsoft""","""Windows Server 2019""",3263
"""Microsoft""","""Windows Server 2019 (Server Co…",3167
"""Microsoft""","""Windows Server 2016""",2954
…,…,…
"""Google Inc.""","""Android""",960
"""Adobe""","""Adobe Experience Manager""",924
"""Microsoft""","""Windows 7""",922


## Providers

In [20]:
# Load CVE dataframe (ensure path is correct relative to this notebook)
df = pl.read_parquet("../www/static/gen/data/test.parquet")

display(df)

# Count unique CVEs per cna_provider
cve_per_cna = (
    df
    .group_by("cna_provider")
    .agg(pl.col("id").n_unique().alias("cve_count"))
    .sort("cve_count", descending=True)
)

# Display and export
display(cve_per_cna)

# only display rows where assigner and cna_provider columns are different
diff_assigner_cna = df.filter(pl.col("assigner") != pl.col("cna_provider"))
display(diff_assigner_cna)


id,assigner,state,cvss_v2,cvss_v3,cvss_v3_1,cvss_v4,adp_cvss_v2,adp_cvss_v3,adp_cvss_v3_1,adp_cvss_v4,date_reserved,date_published,cna_provider
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str
"""CVE-1999-0001""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""2000-02-04T05:00:00""","""mitre"""
"""CVE-1999-0002""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""1999-09-29T04:00:00""","""mitre"""
"""CVE-1999-0003""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""1999-09-29T04:00:00""","""mitre"""
"""CVE-1999-0004""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""2000-02-04T05:00:00""","""mitre"""
"""CVE-1999-0005""","""mitre""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""1999-06-07T00:00:00""","""1999-09-29T04:00:00""","""mitre"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CVE-2025-9994""","""certcc""","""PUBLISHED""",null,null,null,null,null,null,9.8,null,"""2025-09-04T15:31:44.614Z""","""2025-09-09T13:01:05.384Z""","""certcc"""
"""CVE-2025-9996""","""schneider""","""PUBLISHED""",null,null,null,5.8,null,null,null,null,"""2025-09-04T16:16:03.592Z""","""2025-09-09T21:11:15.034Z""","""schneider"""
"""CVE-2025-9997""","""schneider""","""PUBLISHED""",null,null,null,5.8,null,null,null,null,"""2025-09-04T16:16:04.091Z""","""2025-09-09T21:12:35.969Z""","""schneider"""


cna_provider,cve_count
str,u32
"""mitre""",113294
"""Patchstack""",13927
"""microsoft""",12453
"""redhat""",11941
"""VulDB""",11647
…,…
"""securin""",1
"""ELAN""",1
"""FERMAX""",1


id,assigner,state,cvss_v2,cvss_v3,cvss_v3_1,cvss_v4,adp_cvss_v2,adp_cvss_v3,adp_cvss_v3_1,adp_cvss_v4,date_reserved,date_published,cna_provider
str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,str,str,str
"""CVE-2020-12069""","""mitre""","""PUBLISHED""",null,null,7.8,null,null,null,null,null,"""2020-04-22T00:00:00.000Z""","""2022-12-26T00:00:00.000Z""","""CERTVDE"""
"""CVE-2021-38388""","""LINE""","""PUBLISHED""",null,null,8.8,null,null,null,null,null,"""2021-08-10T00:00:00""","""2021-09-08T17:50:45""","""LY-Corporation"""
"""CVE-2021-38487""","""icscert""","""PUBLISHED""",null,null,8.2,8.8,null,null,null,null,"""2021-08-10T00:00:00.000Z""","""2022-05-05T15:18:41.000Z""","""RTI"""
"""CVE-2022-1509""","""@huntrdev""","""PUBLISHED""",null,null,9.9,null,null,null,null,null,"""2022-04-28T00:00:00""","""2022-04-28T10:05:09""","""@huntr_ai"""
"""CVE-2022-21546""","""oracle""","""PUBLISHED""",null,null,null,null,null,null,null,null,"""2021-11-15T19:29:08.898Z""","""2025-05-02T21:52:09.864Z""","""Linux"""
…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""CVE-2025-57789""","""mitre""","""PUBLISHED""",null,null,null,5.3,null,null,null,null,"""2025-08-19T18:25:57.338Z""","""2025-08-20T03:22:08.764Z""","""Commvault"""
"""CVE-2025-57790""","""mitre""","""PUBLISHED""",null,null,null,8.7,null,null,null,null,"""2025-08-19T18:25:57.338Z""","""2025-08-20T03:22:10.697Z""","""Commvault"""
"""CVE-2025-57791""","""mitre""","""PUBLISHED""",null,null,null,6.9,null,null,null,null,"""2025-08-19T18:25:57.338Z""","""2025-08-20T03:22:12.633Z""","""Commvault"""
